In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification
from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead, create_reference_model
from transformers import DataCollatorForLanguageModeling
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from tqdm import tqdm
import os # Import os module to check for model paths

# --- Configuration and Model Loading ---

# Define paths for fine-tuned SFT model and reward model
# IMPORTANT: These paths assume the models are already present locally.
# If not, you will need to download or train them first.
sft_model_path = "sft_model_optimized"
reward_model_path = "reward_model_optimized"

# Check if model paths exist before attempting to load
if not os.path.exists(sft_model_path):
    print(f"Error: SFT model path '{sft_model_path}' not found.")
    print("Please ensure your fine-tuned SFT model is available at this location.")
    exit() # Exit if model not found

if not os.path.exists(reward_model_path):
    print(f"Error: Reward model path '{reward_model_path}' not found.")
    print("Please ensure your reward model is available at this location.")
    exit() # Exit if model not found

# Initialize tokenizer from the SFT model path
print(f"Loading tokenizer from {sft_model_path}...")
tokenizer = AutoTokenizer.from_pretrained(sft_model_path)
# Set the padding token to the end-of-sequence token for consistency
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Tokenizer loaded.")

# Load models onto the GPU if available, otherwise CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

print(f"Loading SFT model from {sft_model_path}...")
sft_model = AutoModelForCausalLM.from_pretrained(sft_model_path).to(device)
print("SFT model loaded.")

print(f"Loading Reward model from {reward_model_path}...")
reward_model = AutoModelForSequenceClassification.from_pretrained(reward_model_path).to(device)
print("Reward model loaded.")

# Load PPO model with a value head for reinforcement learning
# This model is initialized from the SFT model
print(f"Loading PPO model with value head from {sft_model_path}...")
ppo_model = AutoModelForCausalLMWithValueHead.from_pretrained(sft_model_path).to(device)
print("PPO model loaded.")

# Create a reference model for PPO, which is a frozen copy of the PPO model
# used to calculate the KL divergence penalty
print("Creating reference model...")
ref_model = create_reference_model(ppo_model)
print("Reference model created.")

# --- ChromaDB Setup for RAG ---

# Initialize SentenceTransformer for embeddings, used by ChromaDB
print("Initializing SentenceTransformer embedding function...")
embedding_fn = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
print("Embedding function initialized.")

# Initialize ChromaDB client and get/create a collection
print("Setting up ChromaDB client and collection...")
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="rag_chunks", embedding_function=embedding_fn)
print("ChromaDB collection 'rag_chunks' ready.")

# --- RAG Helper Functions ---

def retrieve_chunks(query, k=5):
    """
    Retrieves relevant text chunks from ChromaDB based on a query.

    Args:
        query (str): The user's question or query.
        k (int): The number of top-k relevant chunks to retrieve.

    Returns:
        list: A list of retrieved document chunks.
    """
    print(f"Retrieving {k} chunks for query: '{query}'...")
    results = collection.query(query_texts=[query], n_results=k)
    # Ensure results are not empty and return the documents
    if results and results.get("documents") and results["documents"][0]:
        print(f"Retrieved {len(results['documents'][0])} chunks.")
        return results["documents"][0]
    print("No chunks found for the query.")
    return []

def build_rag_prompt(query, chunks):
    """
    Constructs a RAG-style prompt by combining context chunks and the user's question.

    Args:
        query (str): The user's question.
        chunks (list): A list of retrieved text chunks.

    Returns:
        str: The formatted RAG prompt.
    """
    context = "\n".join(chunks)
    # Format the prompt with context and the user's question
    prompt = f"Context:\n{context}\n\nUser Question:\n{query}\n\nAnswer:"
    print("RAG prompt built.")
    return prompt

# --- Reward Calculation Function ---

def compute_reward(text):
    """
    Computes the reward score for a given text using the reward model.

    Args:
        text (str): The text to evaluate.

    Returns:
        float: The reward score.
    """
    print("Computing reward...")
    # Tokenize the input text, ensuring it's moved to the correct device
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    with torch.no_grad(): # Disable gradient calculation for inference
        # Get the logits from the reward model and squeeze to get a single score
        output = reward_model(**inputs).logits.squeeze(-1)
    print(f"Reward computed: {output.cpu().item():.4f}")
    return output.cpu().item() # Move the result back to CPU and return as a Python float

# --- PPO Trainer Setup ---

# PPO (Proximal Policy Optimization) configuration
ppo_config = PPOConfig(
    model_name=sft_model_path, # Name of the base model
    learning_rate=1.41e-5,     # Learning rate for the optimizer
    batch_size=4,              # Number of samples per batch
    mini_batch_size=2,         # Number of samples per mini-batch for gradient updates
    ppo_epochs=4,              # Number of PPO epochs to run per step
    gradient_accumulation_steps=4, # Accumulate gradients over multiple steps
    target_kl=0.1,             # KL divergence target for PPO
    use_score_scaling=True,    # Whether to scale rewards
    use_score_norm=True        # Whether to normalize rewards
)
print("PPO configuration loaded.")

# Initialize the PPOTrainer
print("Initializing PPOTrainer...")
ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=ppo_model,
    ref_model=ref_model,
    tokenizer=tokenizer,
    # Data collator for language modeling, no masked language model objective
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    # Optimizer for the PPO model
    optimizer=torch.optim.AdamW(ppo_model.parameters(), lr=ppo_config.learning_rate)
)
print("PPOTrainer initialized.")

# --- Example Queries for Training ---

# A list of example user queries to be used for PPO training
queries = [
    "What is artificial intelligence?",
    "Explain photosynthesis.",
    "Describe renewable energy sources.",
    "What is the greenhouse effect?"
]
print(f"Loaded {len(queries)} example queries for training.")

# --- Text Generation Arguments ---

# Keyword arguments for text generation during PPO training
generation_kwargs = {
    "max_new_tokens": 100,        # Maximum number of new tokens to generate
    "do_sample": True,            # Whether to use sampling (True for more diverse outputs)
    "top_p": 0.9,                 # Nucleus sampling parameter
    "temperature": 0.7,           # Controls randomness of predictions
    "pad_token_id": tokenizer.pad_token_id, # ID of the padding token
    "eos_token_id": tokenizer.eos_token_id  # ID of the end-of-sequence token
}
print("Text generation arguments defined.")

# --- PPO Training Loop ---

print("🚀 Starting PPO training...")

# The training loop iterates through the queries multiple times (400 steps total)
for step, query in enumerate(tqdm(queries * 100, desc="PPO steps")):
    print(f"\n--- PPO Step {step} ---")
    # 1. Retrieve relevant chunks for the current query
    chunks = retrieve_chunks(query)
    if not chunks:
        print(f"Skipping step {step} due to no retrieved chunks.")
        continue # Skip to the next step if no chunks are found

    # 2. Build the RAG prompt
    prompt = build_rag_prompt(query, chunks)

    # 3. Tokenize the prompt and move to device
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

    # 4. Generate a response from the PPO model
    print("Generating response...")
    with torch.no_grad():
        # Generate a response using the PPO trainer's generate method
        response = ppo_trainer.generate(input_ids, **generation_kwargs)
    print("Response generated.")

    # 5. Decode the full text (prompt + response)
    # Decode the original prompt and the generated response, then combine them
    full_text = tokenizer.decode(input_ids[0], skip_special_tokens=True) + " " + \
                tokenizer.decode(response[0], skip_special_tokens=True)
    print(f"Full text for reward calculation:\n{full_text}")

    # 6. Compute the reward for the generated full text
    reward = compute_reward(full_text)

    # 7. Perform a PPO training step
    print("Performing PPO step...")
    # The ppo_trainer.step method updates the model based on the generated responses and rewards
    stats = ppo_trainer.step([input_ids], [response], [torch.tensor([reward]).to(device)])
    print("PPO step complete.")

    # 8. Print progress every 50 steps
    if step % 50 == 0:
        print(f"Step {step}, reward: {reward:.4f}")
        # Optionally, print more detailed stats from the PPO trainer
        # print("PPO Stats:", stats)

# --- Save Final Model ---

# Save the final PPO model and tokenizer after training
print("Saving final PPO model and tokenizer...")
ppo_model.save_pretrained("ppo_model_optimized")
tokenizer.save_pretrained("ppo_model_optimized")
print("✅ PPO training complete. Model saved to ppo_model_optimized/")